In [ ]:
import json
import logging

import openeo.processes
from openeo.api.process import Parameter
from openeo.rest.udp import build_process_dict

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
parameters: list[Parameter | dict] = []

In [ ]:
spatial_extent = {
    "west": 30.5503711040000994,
    "south": 1.0709279050000799,
    "east": 31.2229521229999989,
    "north": 1.5469373050000299,
}
temporal_extent = "2020-01-01"

In [ ]:
spatial_extent = Parameter.spatial_extent(
    name="spatial_extent",
    default=spatial_extent
)

parameters.append(spatial_extent)

In [ ]:
canopy_cover_threshold = Parameter.number(
    name="canopy_cover_threshold",
    description="Minimum canopy cover to be considered forest. units: %",
    default=30,
)

parameters.append(canopy_cover_threshold)

In [ ]:
natural_forest_threshold = Parameter.number(
    name="natural_forest_threshold",
    description="Minimum likelihood to be considered natural forest. units: fraction",
    default=0.08,
)

parameters.append(natural_forest_threshold)

In [ ]:
min_connected_area = Parameter.number(
    name="min_connected_area",
    description="Minimum connected area to be considered forest. units: m^2",
    default=10000,
)

parameters.append(min_connected_area)

# UDP

In [ ]:
# TODO: update this to permanent location
natural_forest_stac = "https://s3.waw3-2.cloudferro.com/swift/v1/leon-p6/natural-forest-float32/item.json"

In [ ]:
# Natural Forests of the World 2020
# EPSG:32636 = UTM zone 36N
# dims: ['x', 'y', 'bands']
natural_forest = connection.load_stac(
    url=natural_forest_stac,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["B0"],
)

In [ ]:
# connection.describe_collection("ESA_WORLDCOVER_10M_2020_V1")

In [ ]:
# https://openeofed.dataspace.copernicus.eu/?discover=0&collection=ESA_WORLDCOVER_10M_2020_V1
# 10 m resolution
# EPSG:4326
# openEO backend: terrascope
esa_worldcover = connection.load_collection(
    "ESA_WORLDCOVER_10M_2020_V1",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["MAP"],
)

In [ ]:
# remove the time dimension
esa_worldcover = esa_worldcover.reduce_dimension("t", reducer=openeo.processes.first)

In [ ]:
# connection.describe_collection("CLMS_TCD_PANTROPICAL_10M_YEARLY_V1")

In [ ]:
# tree cover density
# https://openeofed.dataspace.copernicus.eu/?discover=0&collection=CLMS_TCD_PANTROPICAL_10M_YEARLY_V1
# 10 m resolution
# EPSG:4326
# openEO backend: cdse
tree_cover_density = connection.load_collection(
    "CLMS_TCD_PANTROPICAL_10M_YEARLY_V1",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["map"],
)

In [ ]:
# remove the time dimension
tree_cover_density = tree_cover_density.reduce_dimension(
    "t", reducer=openeo.processes.first
)

In [ ]:
# align all 3 datasets in UTM zone 36N
esa_worldcover = esa_worldcover.resample_cube_spatial(natural_forest, method="near")
tree_cover_density = tree_cover_density.resample_cube_spatial(
    natural_forest, method="near"
)

In [ ]:
# make sure band dimension has consistent labels
esa_worldcover = esa_worldcover.rename_labels(dimension="bands", target=["B0"])
tree_cover_density = tree_cover_density.rename_labels(dimension="bands", target=["B0"])

In [ ]:
# cannot use band math with Parameters 🙁
# https://forum.dataspace.copernicus.eu/t/udp-parameter-not-applied/5282

gt_canopy_cover_threshold = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gt",
        "argument": canopy_cover_threshold,
    },
)

gt_natural_forest_threshold = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gt",
        "argument": natural_forest_threshold,
    },
)

In [ ]:
# mask, 1 = natural forest
forest_baseline = (
    (
        esa_worldcover == 10  # class 10 = tree cover
    )
    & (tree_cover_density.apply(gt_canopy_cover_threshold))
    & (
        tree_cover_density <= 100  # values above 100 = unclassifiable / no_data
    )
    & (natural_forest.apply(gt_natural_forest_threshold))
)

In [ ]:
connectivity_udf = openeo.UDF.from_file(
    "../udf/connectivity_mask.py",
    runtime="Python",
    version="3.11",
    context={
        "pixel_area": 10 * 10,
        "min_connected_area": min_connected_area,
    },
)

In [ ]:
# mask where 1 = small region to be excluded
small_region_mask = forest_baseline.apply_neighborhood(
    connectivity_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    # overlap needs to be big enough the reasonably allow for min_pixels
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [ ]:
forest_baseline = forest_baseline & ~small_region_mask

In [ ]:
summary = "construct a natural forest baseline pixel mask for the year 2020"
description = summary

udp_spec = build_process_dict(
    forest_baseline,
    process_id="forest_baseline",
    summary=summary,
    description=description,
    parameters=parameters,
    returns={
        "description": "A pixel mask. 1 = natural forest, 0 = non-forest.",
        "schema": {
            "type": "object",
            "subtype": "datacube"
        }
    },
)

In [ ]:
with open("udp.json", "w") as f:
    json.dump(udp_spec, f, indent=2)